In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path


# ============================================================
# PATHS
# ============================================================

BASE_DIR = Path(r"C:\datacenter_project")

METRICS_FILE = BASE_DIR / "metrics.xlsx"

FIGURES_DIR = BASE_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# LOAD EXCEL FILE
# ============================================================

df = pd.read_excel(METRICS_FILE)

print("Columns found:")
print(df.columns.tolist())

print("\nMetrics file:")
print(df)


# ============================================================
# CONFIGURATION
# ============================================================

windows = [10, 30, 60, 120, 180, 240, 300, 360, 600]

models = {
    "Random Forest": "RF",
    "XGBoost": "XG",
    "Decision Tree": "DT",
    "Linear Regression": "LR"
}


# ============================================================
# FIND METRIC COLUMN
# ============================================================

metric_column = df.columns[0]

# Normalize metric names
df[metric_column] = (
    df[metric_column]
    .astype(str)
    .str.strip()
    .str.upper()
)


# ============================================================
# FUNCTION TO EXTRACT ONE METRIC
# ============================================================

def extract_metric(metric_name):

    row = df[
        df[metric_column] == metric_name.upper()
    ]

    if row.empty:
        raise ValueError(
            f"Metric '{metric_name}' was not found "
            f"in the Excel file."
        )

    row = row.iloc[0]

    data = {}

    for model_name, prefix in models.items():

        values = []

        for window in windows:

            possible_columns = [
                f"{prefix}{window}s",
                f"{prefix}{window}S"
            ]

            found_column = None

            for column in possible_columns:

                if column in df.columns:
                    found_column = column
                    break

            if found_column is None:

                raise ValueError(
                    f"Column for {model_name} "
                    f"{window}s not found."
                )

            values.append(
                float(row[found_column])
            )

        data[model_name] = values

    return pd.DataFrame(
        data,
        index=windows
    )


# ============================================================
# GENERIC GRAPH FUNCTION
# ============================================================

def create_comparison_plot(
    metric_name,
    ylabel,
    filename
):

    metric_df = extract_metric(metric_name)

    plt.figure(figsize=(12, 7))

    # Plot all four algorithms
    for model in models.keys():

        plt.plot(
            metric_df.index,
            metric_df[model],
            marker="o",
            linewidth=2.5,
            markersize=7,
            label=model
        )

    # Axis labels
    plt.xlabel(
        "Observation Window (seconds)",
        fontsize=14
    )

    plt.ylabel(
        ylabel,
        fontsize=14
    )

    # Title
    plt.title(
        f"{metric_name} Comparison Across Algorithms",
        fontsize=18
    )

    # X-axis
    plt.xticks(
        windows,
        [f"{w}s" for w in windows],
        fontsize=12
    )

    # Y-axis
    plt.yticks(fontsize=12)

    # Grid
    plt.grid(
        True,
        linestyle="--",
        alpha=0.4
    )

    # Legend
    plt.legend(
        fontsize=12,
        loc="best"
    )

    # Improve layout
    plt.tight_layout()

    # Save
    output_path = FIGURES_DIR / filename

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(
        f"Saved: {output_path}"
    )


# ============================================================
# 1. R² COMPARISON
# ============================================================

create_comparison_plot(
    metric_name="R^2",
    ylabel="R²",
    filename="R2_comparison_across_algorithms.png"
)


# ============================================================
# 2. RMSE COMPARISON
# ============================================================

create_comparison_plot(
    metric_name="RMSE",
    ylabel="RMSE (°C)",
    filename="RMSE_comparison_across_algorithms.png"
)


# ============================================================
# 3. MAE COMPARISON
# ============================================================

create_comparison_plot(
    metric_name="MAE",
    ylabel="MAE (°C)",
    filename="MAE_comparison_across_algorithms.png"
)


# ============================================================
# DONE
# ============================================================

print("\n==============================================")
print("ALL COMPARISON FIGURES GENERATED")
print("==============================================")

print(f"\nFigures saved in:")
print(FIGURES_DIR)

print("\nGenerated files:")

for file in sorted(FIGURES_DIR.glob("*.png")):
    print(" -", file.name)

Columns found:
['Unnamed: 0', 'RF10s', 'RF30s', 'RF60s', 'RF120s', 'RF180s', 'RF240s', 'RF300s', 'RF360s', 'RF600s', 'XG10s', 'XG30s', 'XG60s', 'XG120s', 'XG180s', 'XG240s', 'XG300s', 'XG360s', 'XG600s', 'DT10s', 'DT30s', 'DT60s', 'DT120s', 'DT180s', 'DT240s', 'DT300s', 'DT360s', 'DT600s', 'LR10s', 'LR30S', 'LR60S', 'LR120S', 'LR180S', 'LR240S', 'LR300S', 'LR360S', 'LR600s']

Metrics file:
  Unnamed: 0   RF10s  RF30s  RF60s  RF120s  RF180s  RF240s  RF300s  RF360s  \
0        R^2   0.603  0.676   0.70   0.722   0.771   0.822   0.841   0.868   
1        MAE   6.910  6.010   5.76   5.230   4.730   3.950   3.510   3.070   
2       RMSE  10.460  9.440   9.09   8.740   7.950   7.000   6.620   6.030   

   RF600s  ...  DT600s  LR10s  LR30S   LR60S  LR120S  LR180S  LR240S  LR300S  \
0   0.917  ...   0.859   0.55   0.59   0.616   0.639   0.666   0.698   0.725   
1   2.380  ...   3.160   8.38   7.98   7.560   7.070   6.630   6.210   5.840   
2   4.790  ...   6.210  11.12  10.61  10.270   9.950  